In [1]:
%load_ext autoreload
%autoreload 2

import scanpy as sc
import sys
import os
import anndata
import pandas as pd
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.pyplot import rc_context
import anndata
import seaborn as sns
import torch
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

print("CUDA Available: ", torch.cuda.is_available())
print("Number of GPUs: ", torch.cuda.device_count())

CUDA Available:  True
Number of GPUs:  1


In [ ]:
data = sc.read_h5ad("../../../original_data/plant_all_days.h5ad")

In [3]:
print(data)
group_map = {'Nodule (21 dpi)': 21, 'Nodule (12 dpi)': 12, 'Root': 0}
data.obs['day'] = data.obs['Sample_time'].map(group_map)

# print(data.obs[['Sample_time', 'day']].drop_duplicates())

AnnData object with n_obs × n_vars = 26712 × 39337
    obs: 'n_genes', 'n_counts', 'Cluster', 'leiden_R', 'Cell type', 'Sample_time'
    var: 'n_cells', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'highly_variable_nbatches', 'mean', 'std', 'SNF', 'means_ForPickMock'
    uns: 'Cell type_colors'
    obsm: 'X_scvi', 'X_umap'
    layers: 'empty', 'normalize_log', 'raw'
    obsp: 'connectivities', 'distances'


In [4]:
# select day == 12
data_group12 = data[data.obs["day"] == 12].copy()

print("day 12 shape:", data_group12.shape)
print("Clusters in day 12:", data_group12.obs["Cluster"].unique())


data_cluster = data_group12[
    data_group12.obs["Cluster"] == '0'
].copy()

print("Cluster:", 0, "shape:", data_cluster.shape)
print("Total cells:", data_cluster.shape[0])
# save h5ad
data_cluster.write_h5ad(
    f"temp/exp3_seperate_train_day12_cluster_0.h5ad"
)


day 12 shape: (8229, 39337)
Clusters in day 12: ['1', '6', '3', '7', '2', ..., '12', '8', '10', '14', '13']
Length: 14
Categories (14, object): ['0', '1', '2', '3', ..., '10', '12', '13', '14']
Cluster: 0 shape: (2026, 39337)
Total cells: 2026


In [15]:
# select day == 21
data_group21 = data[data.obs["day"] == 21].copy()

print("day 21 shape:", data_group21.shape)
print("Clusters in day 21:", data_group21.obs["Cluster"].unique())

data_cluster_7 = data_group21[
    data_group21.obs["Cluster"] == "7"
].copy()
print("Cluster:", 7, "shape:", data_cluster_7.shape)
print("Total cells:", data_cluster_7.shape[0])

data_cluster_11 = data_group21[
    data_group21.obs["Cluster"] == "11"
].copy()
print("Cluster:", 11, "shape:", data_cluster_11.shape)
print("Total cells:", data_cluster_11.shape[0])


# save h5ad
data_cluster_7.write_h5ad(
    f"temp/exp3_seperate_train_day21_cluster_7.h5ad"
)

data_cluster_11.write_h5ad(
    f"temp/exp3_seperate_train_day21_cluster_11.h5ad"
)

day 21 shape: (12004, 39337)
Clusters in day 21: ['2', '12', '9', '4', '0', ..., '6', '10', '3', '13', '14']
Length: 15
Categories (15, object): ['0', '1', '2', '3', ..., '11', '12', '13', '14']
Cluster: 7 shape: (1332, 39337)
Total cells: 1332
Cluster: 11 shape: (554, 39337)
Total cells: 554


In [16]:
# print(len(data_group21.obs["Cluster"].unique()))

In [17]:

train_data_part1 = sc.read_h5ad(f'temp/exp3_seperate_train_day12_cluster_0.h5ad')                                                                                    
print(train_data_part1.obs['day'].unique())
train_data_part2 = sc.read_h5ad(f'temp/exp3_seperate_train_day21_cluster_7.h5ad')
print(train_data_part2.obs['day'].unique())
train_data_part3 = sc.read_h5ad(f'temp/exp3_seperate_train_day21_cluster_11.h5ad')
print(train_data_part2.obs['day'].unique())
combined = anndata.concat([train_data_part1, train_data_part2,train_data_part3])
print(combined.obs['day'].unique)

group_map_main = {'0': 0, '7': 7, '11': 11}
combined.obs['Group'] = combined.obs['Cluster'].map(group_map_main)
print(combined.obs['Group'].unique)
combined.write_h5ad(f'exp3_train_cluster_0_7_11_adata.h5ad')

[12]
Categories (1, int64): [12]
[21]
Categories (1, int64): [21]
[21]
Categories (1, int64): [21]
<bound method Series.unique of AAACGAAAGCGCAATG-1-batch-nodule_small    12
AAACGAAAGTAGGAAG-1-batch-nodule_small    12
AAACGCTCACATGACT-1-batch-nodule_small    12
AAACGCTTCTTGTGCC-1-batch-nodule_small    12
AAAGAACAGGTCATAA-1-batch-nodule_small    12
                                         ..
TTTACGTCAGATACTC-1-batch-nodule_large    21
TTTACTGGTGTGAATA-1-batch-nodule_large    21
TTTCAGTAGCTGTACT-1-batch-nodule_large    21
TTTCAGTAGGTTCATC-1-batch-nodule_large    21
TTTCAGTCATTGAGGG-1-batch-nodule_large    21
Name: day, Length: 3912, dtype: category
Categories (2, int64): [12, 21]>
<bound method Series.unique of AAACGAAAGCGCAATG-1-batch-nodule_small     0
AAACGAAAGTAGGAAG-1-batch-nodule_small     0
AAACGCTCACATGACT-1-batch-nodule_small     0
AAACGCTTCTTGTGCC-1-batch-nodule_small     0
AAAGAACAGGTCATAA-1-batch-nodule_small     0
                                         ..
TTTACGTCAGATACTC-

In [18]:
print("train data Total cells:", combined.shape[0])

train data Total cells: 3912
